# 02 · التنضيف (المرحلة 2)

الهدف: ناخد النسخة الوسخة من الداتا ونطلع منها نسخة نضيفة، **من غير ما نبص على الداتا النضيفة أو سجل المشاكل وقت التنضيف**. بنستخدمهم في الآخر بس عشان نقيس إحنا صلحنا كام.

**قاعدة مهمة:** أي قيمة مفيش مصدر موثوق نرجعها منه (رقم تليفون ناقص مثلًا) بتفضل ناقصة، ومبنخترعش قيم.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd
from src import cleaning, evaluate_cleaning as ev, rules
pd.set_option('display.width', 200); pd.set_option('display.max_rows', 100)
SEED = 42

## 1) الداتا الوسخة وسجل المشاكل

In [2]:
dirty = cleaning.load_dirty(ROOT / 'data/dirty')
log = pd.read_csv(ROOT / 'data/dirty/dirty_issue_log.csv', dtype=str, keep_default_na=False)
print({t: df.shape for t, df in dirty.items()})
log.groupby(['table', 'issue']).size().rename('عدد المشاكل').reset_index()

{'Students': (1002, 12), 'Groups': (41, 8), 'Sessions': (1442, 6), 'Records': (34834, 16), 'Payments': (8713, 9)}


,table,issue,عدد المشاكل
0,Groups,duplicate_row,1
1,Groups,missing_value,7
2,Payments,duplicate_row,17
3,Payments,inconsistent_label,86
4,Payments,missing_value,79
5,Payments,trailing_space,77
6,Records,duplicate_row,69
7,Records,inconsistent_case_or_space,417
8,Records,missing_value,640
9,Records,out_of_range_score,10


## 2) التنضيف

الكود كله في `src/cleaning.py`. كل قرار بيتسجل تلقائيًا وده جدول القرارات:

In [3]:
clean = cleaning.clean_all(dirty)
pd.DataFrame(cleaning.DECISIONS, columns=['الجدول', 'القرار'])

,الجدول,القرار
0,Sessions,حذف الصفوف المكررة تمامًا
1,Sessions,group_id الناقص = من ترقيم session_id (كل مجمو...
2,Sessions,date الناقص = تاريخ السجلات المربوطة بنفس الحصة
3,Sessions,session_number = ترتيب الحصة داخل المجموعة، mo...
4,Sessions,session_type الناقص = makeup لو يوم الحصة مختل...
5,Groups,حذف الصف المكرر
6,Groups,grade و track الناقصين = من اسم المجموعة (أولى...
7,Groups,day = من الاسم أو من يوم حصصها العادية؛ time =...
8,Groups,monthly_fee الناقص = السعر الشائع لنفس (السنة،...
9,Groups,start_date الناقص = تاريخ أول حصة للمجموعة


## 3) مقارنة بالنسخة النضيفة

بنقارن كل عمود بالنسخة النضيفة الأصلية (للتقييم بس).

In [4]:
ref = ev.load_reference(ROOT / 'data/raw/Students_Management_ML_Dataset.xlsx')
cols, shapes = ev.table_report(clean, ref)
shapes

,table,rows_clean,rows_cleaned,same_rows
0,Students,1000,1000,True
1,Groups,40,40,True
2,Sessions,1440,1440,True
3,Records,34765,34765,True
4,Payments,8696,8696,True


In [5]:
print('الأعمدة اللي نسبة تطابقها أقل من 100%:')
cols[cols.match_pct < 100]

الأعمدة اللي نسبة تطابقها أقل من 100%:


,table,column,match_pct,mismatches
0,Students,name,99.90,1
1,Students,phone,99.90,1
2,Students,parent_phone,99.90,1
4,Students,gender,99.90,1
6,Students,school_type,99.90,1
7,Students,area,99.90,1
8,Students,join_date,99.90,1
27,Records,homework,99.93,23
31,Records,notes,99.89,39
37,Records,attendance_notes,99.99,3


## 4) نسبة المشاكل اللي اتصلحت

كل مشكلة في `dirty_issue_log.csv`: اتصلحت لو القيمة بعد التنضيف = القيمة الأصلية.

In [6]:
rep = ev.issue_report(clean, ref, log, dirty)
total = rep.fixed.sum(); n = rep.n.sum()
print(f'اتصلح {total} من {n} مشكلة = {100*total/n:.1f}%')
rep

اتصلح 2086 من 2173 مشكلة = 96.0%


,table,issue,n,fixed,fixed_pct
0,Groups,duplicate_row,1,1,100.0
1,Groups,missing_value,7,7,100.0
2,Payments,duplicate_row,17,17,100.0
3,Payments,inconsistent_label,86,86,100.0
4,Payments,missing_value,79,65,82.3
5,Payments,trailing_space,77,77,100.0
6,Records,duplicate_row,69,69,100.0
7,Records,inconsistent_case_or_space,417,417,100.0
8,Records,missing_value,640,574,89.7
9,Records,out_of_range_score,10,10,100.0


In [7]:
# اللي ما اتصلحش: كله missing_value مفيش مصدر موثوق ليه
rep.assign(not_fixed=rep.n - rep.fixed).query('not_fixed > 0')[['table', 'issue', 'n', 'not_fixed']]

,table,issue,n,not_fixed
4,Payments,missing_value,79,14
8,Records,missing_value,640,66
15,Students,missing_value,10,7


**الخلاصة:** كل المشاكل الشكلية (صفوف مكررة، مسافات، اختلاف حروف، صيغ التليفونات، درجات خارج النطاق، labels الدفع) اتصلحت بنسبة 100%. اللي فضل هو قيم ناقصة فعلًا مفيش ليها مصدر: أسماء وتليفونات وجنس ومنطقة وتاريخ انضمام لطلاب، وتاريخ الدفع، وملاحظات الحصص (نص حر). الـ `homework` وطريقة الدفع الناقصين اتقدّروا بقيمة الطالب المعتادة، فهم تقدير مش استرجاع.

## 5) فحص منطقي: قاعدة الحضور الجديدة

قررنا إن الطالب المتأخر (`late`) بيتحسب حاضر. نشوف الأثر على نسبة الطلاب «في خطر»:

In [8]:
rec = clean['Records'].copy()
rec['attended'] = rules.attended(rec['attendance'])
rec['present_only'] = rec['attendance'].eq('present')
per = rec.groupby('student_id').agg(att_late=('attended', 'mean'), att_present=('present_only', 'mean'), avg=('percentage', 'mean'))
per['risk_late_counts'] = rules.at_risk(per.att_late, per['avg'])
per['risk_present_only'] = rules.at_risk(per.att_present, per['avg'])
print(f"في خطر (late = حضور): {per.risk_late_counts.mean():.1%}   |   في خطر (present بس): {per.risk_present_only.mean():.1%}")
print(f"نسبة الحضور الكلية (late = حضور): {rec.attended.mean():.1%}   |   present بس: {rec.present_only.mean():.1%}")

في خطر (late = حضور): 7.4%   |   في خطر (present بس): 8.3%
نسبة الحضور الكلية (late = حضور): 92.2%   |   present بس: 90.4%


## 6) حفظ الداتا النضيفة

In [9]:
cleaning.save_clean(clean, ROOT / 'data/clean')
sorted(p.name for p in (ROOT / 'data/clean').iterdir())

['Groups_clean.csv',
 'Payments_clean.csv',
 'Records_clean.csv',
 'Sessions_clean.csv',
 'Students_clean.csv']